# **⭕Security Layer**




 This notebook contains the **complete workflow** of the Prompt Safety Classification project.



*   It integrates two models: Logistic Regression and a Deep Learning to classify user prompts as safe or unsafe.

      - LR notbook: https://colab.research.google.com/drive/1kGvHMKytMUvjZ1KhpHNVkHsfoA5lE5Pq?usp=sharing


      - LR HF link: https://huggingface.co/Ranasalh/prompt_safety_2_lr_more_f_and_new_data

      - DL notbook: https://colab.research.google.com/drive/1NAk-aR0LkEW55Sbq1zhPrWKNefQC-NYe?usp=sharing

      - DL HF link: https://huggingface.co/Ranasalh/DLmodel
    



*   The LLM layer (NeutrOn via OpenRouter) is responsible for generating responses only for prompts classified as safe

*   the user interface for interacting with the models and the LLM is built using Streamlit, providing an easy and intuitive way to test and visualize the system’s behavior.
      
      
      
 **This file represents the full logic of the project.**




**Team work** : Rana ALsulami - Danah Alsarrani -  Haitham Alsaloumi -  Ghadi Bakhshwain

















# install Libraries

In [ ]:
!pip install streamlit pyngrok -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 44.6 MB/s eta 0:00:00


In [ ]:
!pip install streamlit


In [ ]:
!pip install pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 120.6 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from pyngrok import ngrok
import os

os.environ["NGROK_AUTH_TOKEN"] = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])

# Backend

In [ ]:
%%writefile models.py
from huggingface_hub import hf_hub_download
import joblib
import numpy as np
import requests
import os
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------
# Config / Paths
# -----------------------------
# ===== BiLSTM Model  =====
model_path_1 = hf_hub_download("Ranasalh/DLmodel", filename="DL_BiLSTM_model.h5")
model_1 = keras.models.load_model(model_path_1)

vec_path_1 = hf_hub_download("Ranasalh/DLmodel", filename="tokenizer.pkl")
vectorizer_1 = joblib.load(vec_path_1)

max_length = 100

# ===== Logistic Regression Model  =====
repo_id_2 = "Ranasalh/prompt_safety_2_lr_more_f_and_new_data"
model_path_2 = hf_hub_download(repo_id=repo_id_2, filename="best_logistic_model.pkl")
vectorizer_path_2 = hf_hub_download(repo_id=repo_id_2, filename="tfidf_vectorizer.pkl")
model_2 = joblib.load(model_path_2)
vectorizer_2 = joblib.load(vectorizer_path_2)

# ===== OpenRouter =====n
os.environ.setdefault("OPEN_ROUTER", "")  # ضع مفتاحك في متغير البيئة OPEN_ROUTER
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODEL = "nvidia/nemotron-nano-12b-v2-vl:free"

# -----------------------------
# Calibration (no retrain) -- Option A
# -----------------------------
# Using a fixed temperature scalar to reduce BiLSTM overconfidence.
# This is a simple, safe approach that doesn't require a validation set
# or retraining. You can tweak TEMPERATURE if you want to adjust
# how aggressively the BiLSTM confidences are pulled down.

TEMPERATURE = 1.8  # قيمة مبدئية جيدة — زدها لتقليل الثقة أكثر، قللها لعكس ذلك
EPS = 1e-8

def apply_temperature_to_sigmoid_prob(prob, temperature=TEMPERATURE):
    """Convert sigmoid prob -> logit -> scale by temperature -> back to prob.

    This approximates temperature scaling for models that output a single
    sigmoid probability.
    """
    prob = float(np.clip(prob, EPS, 1.0 - EPS))
    logit = np.log(prob / (1.0 - prob))
    logit_scaled = logit / float(temperature)
    prob_scaled = 1.0 / (1.0 + np.exp(-logit_scaled))
    return float(prob_scaled)

def normalized_confidence_from_proba_vector(probas):
    """Return a confidence score in [0,1] based on the difference
    between class probabilities. This avoids trusting the raw 'max prob'
    which can be misleading across different model types.
    For binary: |p_pos - p_neg|. For multiclass this generalizes to
    max(p) - second_max(p).
    """
    probs = np.asarray(probas, dtype=float)
    if probs.ndim == 1 and probs.size == 2:
        return float(abs(probs[1] - probs[0]))
    # multiclass fallback: margin between top two
    sorted_idx = np.argsort(probs)
    top = probs[sorted_idx[-1]]
    second = probs[sorted_idx[-2]] if probs.size > 1 else 0.0
    return float(top - second)

# -----------------------------
# Prediction helpers
# -----------------------------
def predict_bilstm_raw_and_calibrated(text):
    """Return (pred_raw, prob_raw, pred_calibrated, prob_calibrated, conf_calibrated)

    - model_1 is expected to output a single sigmoid unit for positive class (UNSAFE).
    - raw prob is the model output; calibrated prob is after temperature scaling.
    - pred values: 0 -> SAFE, 1 -> UNSAFE
    """
    # prepare input (uses the same message formatting you used)
    message = f"Classify the following user prompt --> {text}, as 'SAFE' or 'UNSAFE'. Respond with a single word."
    seq = vectorizer_1.texts_to_sequences([message])
    X1_pad = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')

    # raw prediction: sigmoid output (float in [0,1])
    raw_out = model_1.predict(X1_pad)
    # support shapes: (1,1) or (1,) etc.
    raw_prob = float(np.squeeze(raw_out))
    raw_pred = 1 if raw_prob >= 0.5 else 0

    # calibrated using temperature scalar (no retraining)
    prob_cal = apply_temperature_to_sigmoid_prob(raw_prob, TEMPERATURE)
    pred_cal = 1 if prob_cal >= 0.5 else 0

    # confidence measure for BiLSTM: margin between positive/negative
    # for binary sigmoid we can derive negative prob = 1 - prob
    conf_cal = abs(prob_cal - (1.0 - prob_cal))  # equals |2*p-1|

    return {
        "raw_pred": raw_pred,
        "raw_prob": raw_prob,
        "pred_cal": pred_cal,
        "prob_cal": prob_cal,
        "conf_cal": conf_cal,
    }

def predict_logistic_raw_and_normalized(text):
    """Return (pred, prob_vector, conf_normalized)
    - model_2.predict_proba returns [p_safe, p_unsafe]
    - we compute normalized confidence as |p_pos - p_neg| (margin)
    """
    message = f"Classify the following user prompt --> {text}, as 'SAFE' or 'UNSAFE'. Respond with a single word."
    X2 = vectorizer_2.transform([message])
    probas2 = model_2.predict_proba(X2)[0]
    pred2 = int(np.argmax(probas2))
    # normalized confidence based on margin
    conf2 = normalized_confidence_from_proba_vector(probas2)
    return {
        "pred": pred2,
        "probas": probas2,
        "conf": conf2,
    }

# -----------------------------
# Ensemble decision logic
# -----------------------------
def classify_with_confidence(prompt):
    """Main entry: returns a dict with both models' decisions and a final fused decision.

    Behavior (Option A - no retrain):
      - BiLSTM: apply temperature scaling to sigmoid output
      - LR: use normalized margin confidence (p_pos - p_neg)
      - If both agree -> average calibrated confidences
      - If disagree -> choose the model with higher calibrated/normalized confidence
    """
    b = predict_bilstm_raw_and_calibrated(prompt)
    l = predict_logistic_raw_and_normalized(prompt)

    pred1 = b["pred_cal"]
    conf1 = b["conf_cal"]

    pred2 = l["pred"]
    conf2 = l["conf"]

    if pred1 == pred2:
        final_pred = int(pred1)
        final_conf = float((conf1 + conf2) / 2.0)
        match_type = "Match"
        winner = "Both (agreement)"
    else:
        # choose the model with higher confidence
        if conf1 > conf2:
            final_pred = int(pred1)
            final_conf = float(conf1)
            match_type = "Mismatch"
            winner = "BiLSTM (higher calibrated confidence)"
        else:
            final_pred = int(pred2)
            final_conf = float(conf2)
            match_type = "Mismatch"
            winner = "Logistic Regression (higher normalized confidence)"

    label = "SAFE" if final_pred == 0 else "UNSAFE"

    return {
        "model_1": {"label": "SAFE" if pred1 == 0 else "UNSAFE", "confidence": conf1, "prob_cal": b["prob_cal"], "raw_prob": b["raw_prob"]},
        "model_2": {"label": "SAFE" if pred2 == 0 else "UNSAFE", "confidence": conf2, "probas": l["probas"]},
        "final": {
            "label": label,
            "confidence": final_conf,
            "winner": winner,
            "match_type": match_type,
        },
    }

# -----------------------------
# OpenRouter helper (unchanged)
# -----------------------------
def get_openrouter_response(prompt):
    headers = {
        "Authorization": f"Bearer {os.environ.get('OPEN_ROUTER')}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 200,
    }

    try:
        response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return f"⚠️ Error getting LLM response: {e}"

# -----------------------------
# Public API used by the app
# -----------------------------
def process_prompt(prompt):
    result = classify_with_confidence(prompt)

    conf1 = result['model_1']['confidence']
    conf2 = result['model_2']['confidence']
    final_conf = result['final']['confidence']

    msg = (
        f"Model 1 (BiLSTM): {result['model_1']['label']} "
        f"(calibrated confidence {conf1:.3f}, prob {result['model_1']['prob_cal']:.3f}, raw {result['model_1']['raw_prob']:.3f})\n"
        f"Model 2 (Logistic): {result['model_2']['label']} (normalized confidence {conf2:.3f})\n"
        f"Match type: {result['final']['match_type']}\n\n"
        f"Final decision: {result['final']['label']} "
        f"(chosen from {result['final']['winner']}, confidence {final_conf:.3f})"
    )
    if result['final']['label'] == 'UNSAFE':
       msg = "\n\n⚠️ The LLM refused to answer due to safety concerns, so the response was blocked."


    # Only call LLM if SAFE
    if result['final']['label'] == 'SAFE':
        ai_response = get_openrouter_response(prompt)

        # list of refusal patterns
        refusal_keywords = [
            "i cannot", "i can't", "cannot provide", "i am unable",
            "i’m unable", "i cannot help", "cannot assist", "i won't"
        ]

        # check if model refused
        if any(key in ai_response.lower() for key in refusal_keywords):
            msg = "\n\n⚠️ The LLM refused to answer due to safety concerns, so the response was blocked."
        else:
            msg = f"\n\nResponse:\n{ai_response}"

    return msg

#Frontend

In [ ]:
%%writefile app.py
import streamlit as st
from models import process_prompt
from datetime import datetime
import os
import re

# === Restored API key as requested (user provided) ===
os.environ["OPENROUTER_API_KEY"] = ""  # حطوا الكي حقكممم

# --- Try optional spellchecker ---
try:
    from spellchecker import SpellChecker
    SPELL_AVAILABLE = True
    spell = SpellChecker()
except Exception:
    SPELL_AVAILABLE = False
    spell = None

# --- Constants ---
DARK_BG_COLOR = "#aeaeae"
DARK_TEXT_COLOR = "#ffffff"
INPUT_BG_COLOR = "#003366"
INPUT_TEXT_COLOR = "white"
PRIMARY_COLOR = "#003366"

# --- Page setup ---
st.set_page_config(page_title="Prompt Classifier Chatbot", page_icon="🤖", layout="centered")

# --- Session state ---
ms = st.session_state
if "messages" not in ms:
    ms.messages = []

if "themes" not in ms:
    ms.themes = {
        "current_theme": "light",
        "refreshed": True,
        "light": {
            "theme.base": "light",
            "theme.backgroundColor": "#FFFFFF",
            "theme.primaryColor": PRIMARY_COLOR,
            "theme.secondaryBackgroundColor": "#F1F0F0",
            "theme.textColor": "#000000",
        },
        "dark": {
            "theme.base": "light",
            "theme.backgroundColor": DARK_BG_COLOR,
            "theme.primaryColor": PRIMARY_COLOR,
            "theme.secondaryBackgroundColor": PRIMARY_COLOR,
            "theme.textColor": DARK_TEXT_COLOR,
        },
    }

# --- Helper functions ---

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M")

def detect_safety_tag(text: str) -> str:
    if not text:
        return ""
    t = text.lower()
    unsafe_keys = ["unsafe", "mismatch", "review", "block", "danger", "not allowed", "forbidden"]
    for k in unsafe_keys:
        if k in t:
            return "unsafe"
    safe_keys = ["safe", "allowed", "approved"]
    for k in safe_keys:
        if k in t and "unsafe" not in t:
            return "safe"
    return ""

def safety_emoji(tag: str) -> str:
    if tag == "safe":
        return " 🛡"
    if tag == "unsafe":
        return " ⚠"
    return ""

def simple_spell_suggest(text: str) -> str:
    if not SPELL_AVAILABLE or not spell:
        return text
    parts = re.findall(r"\w+|\W+", text, flags=re.UNICODE)
    corrected_parts = []
    for p in parts:
        if re.match(r"^\w+$", p):
            w = p
            if w.isdigit():
                corrected_parts.append(w)
                continue
            if w.lower() in spell:
                corrected_parts.append(p)
            else:
                corr = spell.correction(w)
                if corr and corr.lower() != w.lower():
                    if w.istitle():
                        corr = corr.title()
                    elif w.isupper():
                        corr = corr.upper()
                    corrected_parts.append(corr)
                else:
                    corrected_parts.append(p)
        else:
            corrected_parts.append(p)
    return "".join(corrected_parts)

def format_message(msg, role):
    time_str = msg["time"].strftime("%H:%M")
    if role == "user":
        return f"""
        <div style="display:flex; justify-content:flex-end; margin:5px 0;">
            <div style="background-color:#ADD8E6; color:black; padding:10px; border-radius:10px; max-width:80%; display:flex; align-items:center; direction:ltr;">
                <span style="color:{PRIMARY_COLOR}; margin-right:5px;">👤</span>
                <span>{msg['content']}</span>
                <span style="font-size:0.7em;color:#555; margin-left:5px;">[{time_str}]</span>
            </div>
        </div>
        """
    else:
        tag = detect_safety_tag(msg.get("content",""))
        emoji = safety_emoji(tag)
        return f"""
        <div style="display:flex; justify-content:flex-start; margin:5px 0;">
            <div style="background-color:{PRIMARY_COLOR}; color:white; padding:10px; border-radius:10px; max-width:80%; display:flex; align-items:center;">
                <span style="color:white; margin-right:5px;">🤖{emoji}</span>
                <span>{msg['content']}</span>
                <span style="font-size:0.7em;color:#ccc; margin-left:5px;">[{time_str}]</span>
            </div>
        </div>
        """

def new_chat():
    ms.messages = []
    st.rerun()

def apply_theme(theme_name):
    tdict = ms.themes[theme_name]
    for key, val in tdict.items():
        if key.startswith("theme"):
            st._config.set_option(key, val)
    ms.themes["current_theme"] = theme_name
    ms.themes["refreshed"] = False

# --- Sidebar ---
with st.sidebar:
    st.markdown("## ⚙ Settings")
    if st.button("🔄 New Chat", use_container_width=True):
        new_chat()
    st.markdown("---")
    theme_choice = st.selectbox(
        "Select Theme",
        ("Light", "Dark"),
        index=0 if ms.themes["current_theme"]=="light" else 1
    )
    if theme_choice.lower() != ms.themes["current_theme"]:
        apply_theme(theme_choice.lower())
        st.rerun()
    st.markdown("---")
    st.markdown(f"**Current Theme:** {ms.themes['current_theme'].capitalize()}")

# --- CSS for themes ---
if ms.themes["current_theme"] == "dark":
    st.markdown(f"""
    <style>
    .stApp {{ background-color:{DARK_BG_COLOR} !important; color:{DARK_TEXT_COLOR} !important;}}
    .stChatInput > div {{ background-color:{INPUT_BG_COLOR} !important; border-radius:0.5rem; }}
    .stChatInput input {{ color:{INPUT_TEXT_COLOR} !important; background-color:{INPUT_BG_COLOR} !important; border: none !important; }}
    div[data-testid="stToolbar"], footer, .css-1n76c6l, div.css-1q8t5l3 {{
        background-color: {DARK_BG_COLOR} !important;
    }}
    </style>
    """, unsafe_allow_html=True)
else:
    st.markdown("""
    <style>
    .stApp { background-color:#FFFFFF !important; color:#000000 !important;}
    .stChatInput > div { background-color:#F1F0F0 !important; border-radius:0.5rem;}
    .stChatInput input { color:black !important; background-color:transparent !important;}
    </style>
    """, unsafe_allow_html=True)

# --- Header ---
st.markdown(f"<h1 style='text-align:center;color:{PRIMARY_COLOR};'>🤖 Chatbot</h1>", unsafe_allow_html=True)
st.markdown(f"<p style='text-align:center;color:#555;'>Secure LLM</p>", unsafe_allow_html=True)

# --- Chat input ---
prompt = st.chat_input("Type your message here...")
if prompt:
    ms.messages.append({"role":"user", "content": prompt, "time": datetime.now()})
    try:
        response = process_prompt(prompt)
    except Exception as e:
        response = f"⚠ Error processing prompt:\n\n{e}"
    ms.messages.append({"role":"assistant", "content": response, "time": datetime.now()})

# --- Display messages + edit panel ---
for i, msg in enumerate(ms.messages):
    if msg["role"] == "user":
        st.markdown(format_message(msg, "user"), unsafe_allow_html=True)
        cols = st.columns([1,8])
        with cols[0]:
            fix_key = f"fix_btn_{i}"
            if st.button("✏", key=fix_key, help="Fix spelling / edit message"):
                ms[f"edit_open_{i}"] = True
        with cols[1]:
            st.write("")
        if ms.get(f"edit_open_{i}", False):
            suggested = simple_spell_suggest(msg.get("content",""))
            input_key = f"edit_input_{i}"
            corrected = st.text_input("Edit & resend:", value=suggested, key=input_key)
            resend_key = f"resend_{i}"
            if st.button("Resend corrected", key=resend_key):
                ms.messages.append({"role":"user", "content": corrected, "time": datetime.now()})
                try:
                    resp = process_prompt(corrected)
                except Exception as e:
                    resp = f"⚠ Error processing prompt:\n\n{e}"
                ms.messages.append({"role":"assistant", "content": resp, "time": datetime.now()})
                ms[f"edit_open_{i}"] = False
                st.rerun()
    else:
        st.markdown(format_message(msg, "assistant"), unsafe_allow_html=True)


Overwriting app.py


In [ ]:
!pip install transformers torch streamlit pyngrok openai --quiet

from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw("streamlit run app.py --server.port 8501 &")
public_url = ngrok.connect(8501)
print("🌍 Public URL:", public_url)

🌍 Public URL: NgrokTunnel: "https://unreiterable-eilene-unretracted.ngrok-free.dev" -> "http://localhost:8501"
